<a href="https://colab.research.google.com/github/venkataratnamb20/quicksilicon/blob/notebooks/vlsi/vbanaloglib/comparator/vbanaloglib_comparator_design.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# vbanaloglib: Analysis and Design of A Comparator

-

## Razavi: The Design of A Comparator

**Comparator design parameters**
- Input offset
- Speed
- Power COnsumption
- Metastability
- Kickback Noise
- Input Referred Electronic Noise

**Aim**

parameter|value|comment
---------|-----|-------
offset|$< 5mV$|-
fclk|5GHz|-



## Strong Arm Latch

- Single CLock Phase
- No static Power
- exhibits an input offset that arises primarily from the input pair
- Delivers rail to rail output swings
- Voltage gain
$$\frac{2g_{m1,2}V_{TH3}}{I_{SS}}$$
- Power Consumption in the signal path
$$Power\ Consumption = 2f_{CLK}C_PV_{DD}^2 + f_{CLK}C_XV_{DD}^2$$
- Power drawn by the `CLK`. $C_{CLK}$ is the total capacitances of the `CLK` net.
$$f_{CLK}C_{CLK}V_{DD}^2$$
-

In [3]:
def get_clk_power(fclk: float, cclk: float, vdd: float) -> float:
    return fclk * cclk * vdd ** 2

def get_power(fclk: float, cp: float, cx:float, vdd: float) -> float:
    return 2*get_clk_power(fclk, cp, vdd) + get_clk_power(fclk, cx, vdd)

fclk = 3e6
cclk = 500e-15
vdd = 2.5
clk_power = get_clk_power(fclk, cclk, vdd)
print(f"Power: {clk_power:.2e}")
cp = 500e-15
cx = 500e-15
total_power = get_power(fclk, cp, cx, vdd)
print(f"Total Power: {total_power:.2e}")


Power: 9.38e-06
Total Power: 2.81e-05


### Design of

- Start with minimum device sizes
- Run in `WORST` case corner.
    - VDD = 1 - 5% = 0.95
    - T = 75deg
    - CLOCK: 50% duty cycle and 10ps rise time.
    - Technology: `CMOS 28nm`
- Begin with transistor dimensions to meet `offset` requirements.
-

## Design for Offset

- The pairs $M_1$ and $M_2$; $M_3$ and $M_4$; $M_5$ and $M_6$ in `Fig 1` appear in the signal path. They must be sized first.

**First design $M_1$ and $M_2$**
- Threshold voltage mismatch. $A_{VTH}$ is constant and is roughly $2.2mV 𝜇m$ in 28nm technology.
$$ΔV_{TH1,2} = \frac{A_{VTH}}{\sqrt{(WL)_{1,2}}}$$

** Choose $M_7$ for enough current.
- Choose M7 (Tail MOS):
M7 should give enough current when $V_{GS7} = VDD$ and $V_{DS7} = V_{in,CM} - V_{GS1,2}$
- Say, $V_{in,CM} = 0.5V$ and $V_{GS1,2} = 0.35V$, We have$V_{DS7} = 0.15V$. The device operates in the deep triode region.
>> Let us select $W_7 = 2μm$ for a current of roughly $0.5mA$
-


In [4]:
from dataclasses import dataclass

@dataclass
class Tech:
    avth: float = 2.2e-3


def get_offset(length_um: float, width_um: float, tech: Tech) -> float:
    return tech.avth / (length * width) ** 0.5

length = 28e-3
width = 10
tech = Tech()
offset = get_offset(length, width, tech)
print(f"Offset: {offset:.2e}")


Offset: 4.16e-03


## Design: input MOS


# Appendix


In [5]:
# !pip install img2table
# !pip install PyMuPDF fitz tools
# !pip install pdftotree


In [6]:
# from img2table.document import Image

# # Instantiation of the image
# img = Image(src="/content/images/tech_params.png")

# # Table identification
# img_tables = img.extract_tables()

# # Result of table identification
# img_tables


In [7]:

# [ExtractedTable(title=None, bbox=(10, 8, 745, 314),shape=(6, 3)),
# ExtractedTable(title=None, bbox=(936, 9, 1129, 111),shape=(2, 2))]

In [8]:
# import fitz  # PyMuPDF
# import pandas as pd

# # Open the PDF file
# pdf_document = fitz.open("/content/data/MARCEL JM PELGROM - ANALOG-TO-DIGITAL CONVERSION-SPRINGER NATURE (2021).pdf")

# # Extract text from each page
# text = ""
# for page_num in range(len(pdf_document)):
#     page = pdf_document.load_page(page_num)
#     text += page.get_text()

# # Assuming the table is in a structured text format, you can use pandas to parse it
# # Example: Convert text to DataFrame (this will vary based on your PDF's structure)
# data = [line.split() for line in text.split('\n') if line]
# df = pd.DataFrame(data)

# # Print the DataFrame
# print(df)


In [9]:
# import fitz  # PyMuPDF
# import pandas as pd

# # Open the PDF file using fitz.open() instead of fitz.open()
# doc = fitz.open("/content/data/MARCEL JM PELGROM - ANALOG-TO-DIGITAL CONVERSION-SPRINGER NATURE (2021).pdf")

# # Extract text from each page
# text = ""
# for page_num in range(len(doc)):
#     page = doc.load_page(page_num)
#     text += page.get_text()

# # Assuming the table is in a structured text format, you can use pandas to parse it
# # Example: Convert text to DataFrame (this will vary based on your PDF's structure)
# data = [line.split() for line in text.split('\n') if line]
# df = pd.DataFrame(data)

# # Print the DataFrame
# print(df)

In [10]:
# import pdftotree

# page = pdftotree.parse('/content/data/MARCEL JM PELGROM - ANALOG-TO-DIGITAL CONVERSION-SPRINGER NATURE (2021).pdf', html_path=None, model_type=None, model_path=None, visualize=False)


In [11]:
!pip install pypdf tabula-py
!pip install jpype1


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 303.4/303.4 kB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 95.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 494.1/494.1 kB 19.5 MB/s eta 0:00:00


In [12]:
import pypdf
from tabula import read_pdf

pdf_file = "/content/data/MARCEL JM PELGROM - ANALOG-TO-DIGITAL CONVERSION-SPRINGER NATURE (2021).pdf"
# pdf_file="https://drive.google.com/uc?export=view&id=19hh6z6p4H7CT-IaY1LU6NcaBlRD0gKtK"
# Get the number of pages in the file
pdf_reader = pypdf.PdfReader(pdf_file)
n_pages = len(pdf_reader.pages)

# For each page the table can be read with the following code
table_pdf = read_pdf(
    pdf_file,
    guess=False,
    pages=181,
    stream=True,
    encoding="utf-8",
    # area=(96, 24, 558, 750),
    columns=(56, 127, 164, 220, 274, 298, 340, 394, 459, 500, 545, 591, 748),
)


In [13]:
table_pdf

[     1                 52   Unnamed: 0          Unnamed: 1         Unnamed: 2  \
 0  NaN    Table 4.8 The v   alues of N         MOS and PMO    S parameters ar   
 1  NaN  technological eff  ects and bi  as settings. See f  or a discussion o   
 2  NaN            Process          VDD                 dox                Cox   
 3  NaN               Unit         Volt                  nm             fF/μm2   
 4  NaN              0.8μm          3.3                  15                2.3   
 5  NaN              0.6μm          3.3                  13                2.7   
 6  NaN              0.5μm          3.3                  12                2.9   
 7  NaN             0.35μm          3.3                 7.7                4.3   
 8  NaN             0.25μm          2.5                   6                6.9   
 9  NaN             0.18μm          1.8                   4                8.3   
 10 NaN             0.13μm          1.2                 2.5                 11   
 11 NaN         

In [14]:

# table_pdf = read_pdf(
#     pdf_file,
#     guess=False,
#     pages=181,
#     stream=True,
#     encoding="utf-8",
# )
# table_pdf

In [15]:
type(table_pdf)

list

In [16]:
table_df_final = table_pdf[0].dropna(axis=1).drop([0, 1])

In [17]:
cols = table_df_final.iloc[:2].T
cols = cols.apply(lambda x: x.str.cat(sep=" "), axis=1)
cols = cols.to_list()
cols
table_df_final.columns = cols
table_df_final = table_df_final.iloc[2:].reset_index(drop=True)
table_df_final

,Process Unit,VDD Volt,dox nm,Cox fF/μm2,"VT ,n Volt","VT ,p Volt","AV T,n mVμm"
0,0.8μm,3.3,15,2.3,0.6,−0.6,10.7
1,0.6μm,3.3,13,2.7,0.65,−0.8,11.0
2,0.5μm,3.3,12,2.9,0.6,−0.6,9
3,0.35μm,3.3,7.7,4.3,0.63,−0.6,8
4,0.25μm,2.5,6,6.9,0.57,−0.5,6
5,0.18μm,1.8,4,8.3,0.48,−0.5,6
6,0.13μm,1.2,2.5,11,0.34,−0.3,5
7,90 nm (LP),1.2,2.3,11.7,0.37,−0.3,4.5
8,65 nm (LP),1.2,2.2,12.6,0.32,−0.3,4.5
9,45 nm (LP),1.1,1.7,16,0.39,−0.4,4


In [18]:
table_df_final

,Process Unit,VDD Volt,dox nm,Cox fF/μm2,"VT ,n Volt","VT ,p Volt","AV T,n mVμm"
0,0.8μm,3.3,15,2.3,0.6,−0.6,10.7
1,0.6μm,3.3,13,2.7,0.65,−0.8,11.0
2,0.5μm,3.3,12,2.9,0.6,−0.6,9
3,0.35μm,3.3,7.7,4.3,0.63,−0.6,8
4,0.25μm,2.5,6,6.9,0.57,−0.5,6
5,0.18μm,1.8,4,8.3,0.48,−0.5,6
6,0.13μm,1.2,2.5,11,0.34,−0.3,5
7,90 nm (LP),1.2,2.3,11.7,0.37,−0.3,4.5
8,65 nm (LP),1.2,2.2,12.6,0.32,−0.3,4.5
9,45 nm (LP),1.1,1.7,16,0.39,−0.4,4


In [24]:
table_df_final[table_df_final['Process Unit'] == '90 nm (LP)']
table_df_final[table_df_final['Process Unit'] == '0.25μm']

,Process Unit,VDD Volt,dox nm,Cox fF/μm2,"VT ,n Volt","VT ,p Volt","AV T,n mVμm"
4,0.25μm,2.5,6,6.9,0.57,−0.5,6


## Comparator: 0.25um


In [36]:
avtn = table_df_final[table_df_final['Process Unit'] == '0.25μm']['AV T,n mVμm']
print(type(avtn))
avtn = float(avtn.values[0])

tech = Tech(avth = avtn)
length = 240e-3
width = 16
print(get_offset(length, width, tech))
# avtn/(length*width)**0.5

<class 'pandas.core.series.Series'>
3.0618621784789726


In [30]:
avtn.values[0]

'6'